# Full stack

Get an API-key!

In [ ]:
API_KEY = "YOUR KEY GOES HERE"

### Sample

We test on samples. This first chunk samples the dataset by selecting all rows (verb-observations) corresponding to a random text id.

In [ ]:
import pandas as pd
import random

df = pd.read_csv("Verbs_ready_for_llm.csv")

groups = df['id'].unique()

sampled_groups = random.sample(list(groups), 1)

df_sample = df[df['id'].isin(sampled_groups)]

df_sample.to_csv("sampled_texts_as_verbs.csv", index=False)

df_sample

,Unnamed: 0,id,date,text,verb,start_char,end_char
4550,4551,ST_CROIX_464,25-07-1805,"Ran away from the Subscriber a few Days ago, H...",Ran,0,3
4551,4552,ST_CROIX_464,25-07-1805,"Ran away from the Subscriber a few Days ago, H...",named,59,64
4552,4553,ST_CROIX_464,25-07-1805,"Ran away from the Subscriber a few Days ago, H...",give,140,144
4553,4554,ST_CROIX_464,25-07-1805,"Ran away from the Subscriber a few Days ago, H...",bring,177,182


### First step: Subject, object and snippet

First we identify the subject, a title for the subject (emic label), the object for the verb, as well as extract the corresponding snippet of text describing the action. An infinitive form of the verb is generated too.

In [ ]:
import dspy
import pandas as pd

lm = dspy.LM("mistral/mistral-large-2512", api_key=API_KEY, num_retries=5)
dspy.configure(lm=lm)

df = pd.read_csv('sampled_texts_as_verbs.csv')

class ExtractActionData(dspy.Signature):
    """Extract semantic action triples from nineteenth-century runaway advertisement.

    Role: You are an expert computational linguist specializing in 19th-century text and
    semantic role labeling.

    You are given a runaway advertisement with ONE specific verb occurrence marked inline using »double angle
    quotes«, e.g. »run«. This marked verb — and ONLY this specific occurrence — is the action
    you must analyze, even if the same verb (or a NER-tokenized/merged form of it) appears elsewhere
    unmarked in the text. Ignore all other occurrences of the same or similar verbs.

    CRITICAL — subject identification: Do NOT default to the advertisement's protagonist as Subject unless they are
    genuinely the one performing THIS marked action. Determine the subject strictly from the local
    clause around the marker."""

    action_info: str | None = dspy.InputField(desc="A runaway advertisement from the newspaper at St Croix with one target verb occurrence wrapped in »guillemets« marking exactly which instance to analyze.")
    Subject: str | None = dspy.OutputField(desc="The semantic subject of the marked verb — the specific entity performing THIS action.")
    Title: str | None = dspy.OutputField(desc="The occupational title, judicial status or other emic label of the semantic subject of the marked verb.")
    Verb: str | None = dspy.OutputField(desc="The full, reconstructed form of the marked verb.")
    Object: str | None = dspy.OutputField(desc="The full semantic object of the marked verb specifically. If none output null.")
    Infinitive: str | None = dspy.OutputField(desc="The infinitive form of the marked verb and ONLY the marked verb.")
    Snippet: str | None = dspy.OutputField(desc="The full sentence containing the marked verb occurrence (without the » « markers in your output). Make no corrections to the original wording.")

classify = dspy.Predict(ExtractActionData)


def mark_target_verb(full_text: str, start: int, end: int) -> str:
    """Insert »...« markers around the exact character span of the target verb."""
    return f"{full_text[:start]}»{full_text[start:end]}«{full_text[end:]}"


results = []
for idx, row in df.iterrows():
    try:
        full_text = row['text']
        start = int(row['start_char'])
        end = int(row['end_char'])

        marked_input = mark_target_verb(full_text, start, end)

        response = classify(action_info=marked_input)
        results.append({
            'Subject': response.Subject,
            'Title': response.Title,
            'Verb': response.Verb,
            'Object': response.Object,
            'Infinitive': response.Infinitive,
            'Snippet': response.Snippet
        })
        print(f"Processed {idx + 1}/{len(df)}")
    except Exception as e:
        print(f"Error on row {idx}: {e}")
        results.append({
            'Subject': None,
            'Title': None,
            'Verb': None,
            'Object': None,
            'Infinitive': None,
            'Snippet': None
        })

results_df = pd.DataFrame(results)
df = pd.concat([df, results_df], axis=1)

df.to_csv('St_croix_verbs_via_mistral_large.csv', index=False)

lm.inspect_history(n=1)

Processed 1/4
Processed 2/4
Processed 3/4
Processed 4/4




[2026-07-07T08:51:57.538907]

System message:

Your input fields are:
1. `action_info` (UnionType[str, NoneType]): A runaway advertisement from the newspaper at St Croix with one target verb occurrence wrapped in »guillemets« marking exactly which instance to analyze.
Your output fields are:
1. `Subject` (UnionType[str, NoneType]): The semantic subject of the marked verb — the specific entity performing THIS action.
2. `Title` (UnionType[str, NoneType]): The occupational title, judicial status or other emic label of the semantic subject of the marked verb.
3. `Verb` (UnionType[str, NoneType]): The full, reconstructed form of the marked verb.
4. `Object` (UnionType[str, NoneType]): The full semantic object of the marked verb specifically. If none output null.
5. `Infinitive` (UnionType[str, NoneType]): The infinitive form of the marked verb and ONLY the marked verb.
6. `Snippet` (UnionType[str, NoneType]): The full sentence con

### Location

Next up: Location, anchored in the snippet.

In [ ]:
import dspy
import pandas as pd

lm = dspy.LM("mistral/mistral-large-2512", api_key=API_KEY, num_retries=5)
dspy.configure(lm=lm)

df = pd.read_csv('St_croix_verbs_via_mistral_large.csv') 

class ExtractActionData(dspy.Signature):
    """Extract the location of an action described in a runaway advertisement c. 1800.
    'Location' includes both proper place names (e.g. city, street, or building names)
    and vague or relational spatial references (e.g. "at her mother's", "in the field",
    "nearby", "at home"). Extract the location as it is actually expressed in the
    snippet — do not invent a specific place name if the source only gives a vague
    or relational description.
    Use the snippet itself as the primary evidence, but use the subject, title,
    verb, and surrounding text as supporting context to resolve ambiguous or
    implicit location references (e.g. pronouns, deictic phrases, or locations
    only named earlier in the source text)."""

    text: str | None = dspy.InputField(desc="The full source text the snippet was drawn from, for context, also including the verb and its character span")
    Subject: str | None = dspy.InputField(desc="The subject performing the action")
    Title: str | None = dspy.InputField(desc="The title or heading associated with this entry")
    Verb: str | None = dspy.InputField(desc="The verb describing the action")
    Snippet: str | None = dspy.InputField(desc="The text snippet describing the action")
    Location: str | None = dspy.OutputField(desc="The location of the action as expressed in the snippet. This may be a proper name (city, street, building) or a vague/relational description (e.g. 'in the field', 'at her mother's', 'outside'). Resolve pronouns or deictic references using context, but preserve the original level of specificity — do not upgrade a vague mention into a named place unless the text itself supports that identification.")

classify = dspy.ChainOfThought(ExtractActionData)

results = []
for idx, row in df.iterrows():
    try:
        response = classify(
            text=row.get('text'),
            Subject=row.get('Subject'),
            Title=row.get('Title'),
            Verb=row.get('Verb'),
            Snippet=row['Snippet'],
        )
        results.append({
            'Location': response.Location,
        })
        print(f"Processed {idx + 1}/{len(df)}")
    except Exception as e:
        print(f"Error on row {idx}: {e}")
        results.append({
            'Location': None,
        })

results_df = pd.DataFrame(results)
df = pd.concat([df, results_df], axis=1)
df.to_csv('St_croix_verbs_via_mistral_large_locations.csv', index=False)

lm.inspect_history(n=1)

Processed 1/4
Processed 2/4
Processed 3/4
Processed 4/4




[2026-07-07T08:51:57.639553]

System message:

Your input fields are:
1. `text` (UnionType[str, NoneType]): The full source text the snippet was drawn from, for context, also including the verb and its character span
2. `Subject` (UnionType[str, NoneType]): The subject performing the action
3. `Title` (UnionType[str, NoneType]): The title or heading associated with this entry
4. `Verb` (UnionType[str, NoneType]): The verb describing the action
5. `Snippet` (UnionType[str, NoneType]): The text snippet describing the action
Your output fields are:
1. `reasoning` (str): 
2. `Location` (UnionType[str, NoneType]): The location of the action as expressed in the snippet. This may be a proper name (city, street, building) or a vague/relational description (e.g. 'in the field', 'at her mother's', 'outside'). Resolve pronouns or deictic references using context, but preserve the original level of specificity — do not upgrade a vague men

### Instruments

The objects used to carry out or prop up actions are derived in a separate pass.

In [ ]:
import dspy
import pandas as pd

lm = dspy.LM("mistral/mistral-large-2512", api_key=API_KEY, num_retries=5)
dspy.configure(lm=lm)

df = pd.read_csv('St_croix_verbs_via_mistral_large_locations.csv')

class ExtractInstrumentData(dspy.Signature):
    """Extract the instrument(s) used to carry out an action described in a 19th-century text snippet.

    An instrument is any object, tool, or document that supports or enables the
    action — for example tools for a work task, papers for an identity inspection,
    coins for a purchase, a weapon for an assault, or a letter for a communication.
    Use the snippet as the primary evidence, but use the subject, title, verb, and
    surrounding text as supporting context to resolve ambiguous or implicit
    references. If multiple distinct instruments are mentioned, list them all,
    separated by commas. If no instrument is mentioned, return None."""

    text: str | None = dspy.InputField(desc="The full source text the snippet was drawn from, for context")
    Subject: str | None = dspy.InputField(desc="The subject performing the action")
    Title: str | None = dspy.InputField(desc="The title or heading associated with this entry")
    Verb: str | None = dspy.InputField(desc="The verb describing the action")
    action_info: str | None = dspy.InputField(desc="The text snippet describing the action")
    Instrument: str | None = dspy.OutputField(desc="The instrument(s) used in the action, as a single string, comma-separated if more than one is mentioned, or None if no instrument is mentioned. Instruments have to be mentioned explicitly.")

classify = dspy.ChainOfThought(ExtractInstrumentData)

results = []
for idx, row in df.iterrows():
    try:
        response = classify(
            text=row.get('text'),
            Subject=row.get('Subject'),
            Title=row.get('Title'),
            Verb=row.get('Verb'),
            action_info=row['Snippet'],
        )
        results.append({
            'Instrument': response.Instrument,
        })
        print(f"Processed {idx + 1}/{len(df)}")
    except Exception as e:
        print(f"Error on row {idx}: {e}")
        results.append({
            'Instrument': None,
        })

results_df = pd.DataFrame(results)
df = pd.concat([df, results_df], axis=1)
df.to_csv('St_croix_verbs_via_api_mistral_instruments.csv', index=False)

Processed 1/4
Processed 2/4
Processed 3/4
Processed 4/4


### Disambiguation of subjects

Subjects might appear in relation to multiple actions. Let's disambiguate their appearances.

In [ ]:
import dspy
import pandas as pd
import json
from pydantic import BaseModel
from typing import List

lm = dspy.LM("mistral/mistral-large-2512", api_key=API_KEY, num_retries=5)
dspy.configure(lm=lm)

df = pd.read_csv('St_croix_verbs_via_api_mistral_instruments.csv') 
class EntityCluster(BaseModel):
    entity_id: int
    canonical_subject: str
    mention_indices: List[int]

class DisambiguateEntities(dspy.Signature):
    """Cluster subject mentions extracted from the same 19th-century text that refer to the same entity. Entities include people, but also objects and immaterial actors. Use Subject, Title, Verb, and Snippet (plus the source text for context) to decide whether two mentions denote the same entity, even if named slightly differently (titles, spelling, partial names). Every mention index must appear in exactly one cluster; singletons are fine."""
    text: str = dspy.InputField(desc="The full source text shared by all mentions in this group")
    mentions: str = dspy.InputField(desc="JSON array of mentions: [{index, Subject, Title, Verb, Snippet}, ...]")
    clusters: List[EntityCluster] = dspy.OutputField(desc="Clusters of mention indices referring to the same entity, each with a unique entity_id and a canonical_subject label")

disambiguate = dspy.ChainOfThought(DisambiguateEntities)

all_groups = []

for text_id, group in df.groupby('id'):
    group = group.reset_index(drop=True)
    mentions_payload = [
        {
            "index": i,
            "Subject": row['Subject'],
            "Title": row['Title'],
            "Verb": row['Verb'],
            "Snippet": row['Snippet'],
        }
        for i, row in group.iterrows()
    ]
    text_value = group['text'].iloc[0]

    index_to_entity, index_to_canon = {}, {}
    try:
        response = disambiguate(text=text_value, mentions=json.dumps(mentions_payload, ensure_ascii=False))
        for cluster in response.clusters:
            for idx in cluster.mention_indices:
                index_to_entity[idx] = cluster.entity_id
                index_to_canon[idx] = cluster.canonical_subject
    except Exception as e:
        print(f"Error on id {text_id}: {e}")

    next_id = (max(index_to_entity.values(), default=-1) + 1)
    for idx in group.index:
        if idx not in index_to_entity:
            index_to_entity[idx] = next_id
            index_to_canon[idx] = group.loc[idx, 'Subject']
            next_id += 1

    group['local_entity_id'] = group.index.map(index_to_entity)
    group['cluster_id'] = group['local_entity_id'].apply(lambda x: f"{text_id}-{x}")
    group['canonical_subject'] = group.index.map(index_to_canon)
    all_groups.append(group)

    print(f"Processed id {text_id} ({len(group)} mentions)")

result_df = pd.concat(all_groups, ignore_index=True)
result_df.to_csv('disambiguated_entities.csv', index=False)

Processed id ST_CROIX_464 (4 mentions)


### Disambiguation of verb phrase objects

The subjects might also appear as objects of sentences, and there might be more entities hiding there too. Here we try to identify and disambiguate those, matching them with canonical subjects.

In [ ]:
import dspy
import pandas as pd
import json
from pydantic import BaseModel
from typing import List, Optional

lm = dspy.LM("mistral/mistral-large-2512", api_key=API_KEY, num_retries=5)
dspy.configure(lm=lm)

class ObjectMatch(BaseModel):
    mention_index: int
    matched_entity_id: Optional[int]  

class MatchObjectsToEntities(dspy.Signature):
    """For each mention, check the Object field, using the full source text as context to resolve references (e.g. pronouns, "the same X", or other back-references). Determine whether the Object mentions/names one of the already-disambiguated entities listed below — i.e. contains a name, title, or clear designation matching an entity's canonical_subject. If the Object does not reference any already listed entity, set matched_entity_id to null."""
    text: str = dspy.InputField(desc="The full source text shared by all mentions in this group, for context")
    entities: str = dspy.InputField(desc="JSON array of disambiguated entities: [{entity_id, canonical_subject}, ...]")
    mentions: str = dspy.InputField(desc="JSON array of mentions to check: [{index, Object}, ...]")
    matches: List[ObjectMatch] = dspy.OutputField(desc="One match result per mention_index, matched_entity_id is the entity_id explicitly named in the Object, or null if none")

class ObjectCluster(BaseModel):
    canonical_object: str
    mention_indices: List[int]

class DisambiguateNewObjects(dspy.Signature):
    """The Object mentions below did NOT match any already-known entity. Cluster these objects so that mentions referring to the same entity (such as a physical object, document, or idea) are grouped together, even if worded differently (e.g. "his papers" and "the papers" referring back to the same documents). Objects include things like tools, papers, coins, or any other item involved in the action. Objects can also be immaterial. Use the full source text for context. Every mention index must appear in exactly one cluster; singletons are fine. Give each cluster a canonical_object label describing what the object is."""
    text: str = dspy.InputField(desc="The full source text shared by all mentions in this group")
    mentions: str = dspy.InputField(desc="JSON array of unmatched object mentions: [{index, Object}, ...]")
    clusters: List[ObjectCluster] = dspy.OutputField(desc="Clusters of mention indices referring to the same new object, each with a canonical_object label")

match_objects = dspy.ChainOfThought(MatchObjectsToEntities)
disambiguate_objects = dspy.ChainOfThought(DisambiguateNewObjects)

df = pd.read_csv('disambiguated_entities.csv') 

all_groups = []
for text_id, group in df.groupby('id'):
    group = group.reset_index(drop=True)

    text_value = group['text'].iloc[0]

    entities_for_text = (
        group[['local_entity_id', 'canonical_subject']]
        .drop_duplicates()
        .rename(columns={'local_entity_id': 'entity_id'})
        .to_dict(orient='records')
    )
    object_payload = [
        {"index": i, "Object": row.get('Object')}
        for i, row in group.iterrows()
    ]

    index_to_object_entity = {}
    index_to_object_canon = {}

    try:
        match_response = match_objects(
            text=text_value,
            entities=json.dumps(entities_for_text, ensure_ascii=False),
            mentions=json.dumps(object_payload, ensure_ascii=False),
        )
        for m in match_response.matches:
            index_to_object_entity[m.mention_index] = m.matched_entity_id
    except Exception as e:
        print(f"Error matching objects on id {text_id}: {e}")

    canon_lookup = dict(zip(group['local_entity_id'], group['canonical_subject']))

    unmatched_indices = [
        i for i in group.index
        if index_to_object_entity.get(i) is None
    ]

    if unmatched_indices:
        unmatched_payload = [
            {"index": i, "Object": group.loc[i, 'Object']}
            for i in unmatched_indices
        ]
        next_id = int(group['local_entity_id'].max()) + 1

        try:
            cluster_response = disambiguate_objects(
                text=text_value,
                mentions=json.dumps(unmatched_payload, ensure_ascii=False),
            )
            for cluster in cluster_response.clusters:
                new_id = next_id
                next_id += 1
                for idx in cluster.mention_indices:
                    index_to_object_entity[idx] = new_id
                    index_to_object_canon[idx] = cluster.canonical_object
        except Exception as e:
            print(f"Error disambiguating new objects on id {text_id}: {e}")
            for idx in unmatched_indices:
                index_to_object_entity[idx] = next_id
                index_to_object_canon[idx] = group.loc[idx, 'Object']
                next_id += 1

    group['object_entity_id'] = group.index.map(lambda i: index_to_object_entity.get(i))
    group['object_cluster_id'] = group['object_entity_id'].apply(lambda x: f"{text_id}-{x}" if pd.notna(x) else None)
    group['object_canonical'] = group.index.map(
        lambda i: canon_lookup.get(index_to_object_entity.get(i))
        if index_to_object_entity.get(i) in canon_lookup
        else index_to_object_canon.get(i)
    )

    all_groups.append(group)
    print(f"Processed id {text_id} ({len(group)} mentions)")

result_df = pd.concat(all_groups, ignore_index=True)
result_df.to_csv('disambiguated_entities_with_objects.csv', index=False)

Processed id ST_CROIX_464 (4 mentions)


### Role labelling

Placing entities in relation to actions using the GRAM-framework.

In [ ]:
import dspy
import pandas as pd
import json
from pydantic import BaseModel
from typing import Optional

lm = dspy.LM("mistral/mistral-large-2512", api_key=API_KEY, num_retries=5)
dspy.configure(lm=lm)

GRAM_ROLES = [
    "HAS_ADDRESSEE", "HAS_AGENT", "HAS_BENEFACTIVE", "HAS_COMITATIVE",
    "HAS_EFFECTOR", "HAS_EXPERIENCER", "HAS_INSTRUMENT", "HAS_MEDIATOR",
    "HAS_MEDIUM", "HAS_PATIENT", "HAS_RECIPIENT", "HAS_RESULT",
    "HAS_TARGET", "HAS_THEME",
]

class RoleLabels(BaseModel):
    subject_role: Optional[str]  
    object_role: Optional[str] 
class LabelSemanticRoles(dspy.Signature):
    """Label the semantic role of the canonical subject and canonical object in the action described by the verb, using the GRAM framework role inventory below. Choose exactly one role per entity (subject and object) from this closed list, based on how that actor participates in the action denoted by the verb in the snippet. If an actor is absent, unclear, or the role inventory does not apply, set its role to null.

    HAS_ADDRESSEE: the entity receives a message through the action and is thereby its audience
    HAS_AGENT: the entity is the agent performing the action itself
    HAS_PATIENT: the entity is the patient of the action (often the semantic object)
    HAS_BENEFACTIVE: the entity derives a benefit from the action but is not its agent
    HAS_COMITATIVE: the entity is performing the action together with others, becoming co-agents of the action
    HAS_EFFECTOR: the entity triggers or causes the action but is not the agent (e.g. the person who orders the action, or an environmental event that causes the action)
    HAS_EXPERIENCER: the entity observes, perceives, or senses the action, but is neither its agent or its addressee (e.g. a feeling)
    HAS_INSTRUMENT: the entity is a tool used in the action
    HAS_MEDIATOR: the entity serves actively as an intermediary between the agent and the patient
    HAS_MEDIUM: the entity is the material means through which mediation is realized (e.g. a contract, an amount of money, a transportation)  
    HAS_RECIPIENT: the entity receives something tangible through the action (as possessor)
    HAS_RESULT: the entity comes into existence through the action
    HAS_TARGET: the entity is the yet unrealized target or ambition of action
    HAS_THEME: the entity is the topic of verbal action (e.g. of a discussion or a narrative)
    """
    verb: str = dspy.InputField(desc="The full verb describing the action")
    snippet: str = dspy.InputField(desc="The text snippet describing the action, for context")
    canonical_subject: Optional[str] = dspy.InputField(desc="The canonical (disambiguated) subject actor, or null if absent")
    canonical_object: Optional[str] = dspy.InputField(desc="The canonical (disambiguated) object actor, or null if absent")
    roles: RoleLabels = dspy.OutputField(desc="The GRAM role of the subject and of the object")

label_roles = dspy.ChainOfThought(LabelSemanticRoles)

df = pd.read_csv('disambiguated_entities_with_objects.csv') 

subject_roles, object_roles = [], []

for idx, row in df.iterrows():
    try:
        response = label_roles(
            verb=row['Verb'],
            snippet=row['Snippet'],
            canonical_subject=row.get('canonical_subject') if pd.notna(row.get('canonical_subject')) else None,
            canonical_object=row.get('object_canonical') if pd.notna(row.get('object_canonical')) else None,
        )
        role = response.roles
        subject_roles.append(role.subject_role)
        object_roles.append(role.object_role)
        print(f"Processed {idx + 1}/{len(df)}")
    except Exception as e:
        print(f"Error on row {idx}: {e}")
        subject_roles.append(None)
        object_roles.append(None)

df['subject_gram_role'] = subject_roles
df['object_gram_role'] = object_roles
df.to_csv('disambiguated_entities_with_roles.csv', index=False)

Processed 1/4
Processed 2/4
Processed 3/4
Processed 4/4


### Place-Action labelling

Figuring out what role a place plays in relation to actions.

In [ ]:
import dspy
import pandas as pd
import json
from pydantic import BaseModel
from typing import Optional

lm = dspy.LM("mistral/mistral-large-2512", api_key=API_KEY, num_retries=5)
dspy.configure(lm=lm)

GRAM_ROLES = [
    "TAKES_PLACE", "HAS_DESTINATION", "HAS_ORIGIN", "IS_SUBJECT_OF",
]

class RoleLabels(BaseModel):
    location_role: Optional[str] 

class LabelSemanticRoles(dspy.Signature):
    """Label the role of the place mentioned as part of a snippet of text describing an action, using the role inventory below. Choose exactly one role per location from this closed list, based on what role the place has in the action denoted by the verb in the snippet. If a location is absent, unclear, or the role inventory does not apply, set its role to null.

    TAKES_PLACE: the location is where the action itself takes place
    HAS_DESTINATION: the location is mentioned as the destination of an action, typically involving mobility
    HAS_ORIGIN: the location is the spatial starting-point of an action that has a destination (that might or might not be mentioned)
    IS_SUBJECT_OF: the location is mentioned as the location of a past, future or hypothetical action
    """
    verb: str = dspy.InputField(desc="The full verb describing the action")
    snippet: str = dspy.InputField(desc="The text snippet describing the action, for context")
    canonical_subject: Optional[str] = dspy.InputField(desc="The canonical (disambiguated) subject actor, or null if absent")
    canonical_object: Optional[str] = dspy.InputField(desc="The canonical (disambiguated) object actor, or null if absent")
    roles: RoleLabels = dspy.OutputField(desc="The GRAM role of the location")

label_roles = dspy.ChainOfThought(LabelSemanticRoles)

df = pd.read_csv('disambiguated_entities_with_roles.csv') 

location_roles = []

for idx, row in df.iterrows():
    try:
        response = label_roles(
            verb=row['Verb'],
            snippet=row['Snippet'],
            canonical_subject=row.get('canonical_subject') if pd.notna(row.get('canonical_subject')) else None,
            canonical_object=row.get('object_canonical') if pd.notna(row.get('object_canonical')) else None,
        )
        role = response.roles
        location_roles.append(role.location_role)
        print(f"Processed {idx + 1}/{len(df)}")
    except Exception as e:
        print(f"Error on row {idx}: {e}")
        location_roles.append(None)

df['location_gram_role'] = location_roles

df.to_csv('disambiguated_entities_with_roles_location.csv', index=False)

Processed 1/4
Processed 2/4
Processed 3/4
Processed 4/4
